<a href="https://colab.research.google.com/github/Pedro-Lucas-Vieira/Aurora-AI/blob/main/base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 ============================================================
# AURORA AI
# Arquivo: criar_base.py
# ============================================================
#
# Objetivo:
#
# Este arquivo cria a base vetorial utilizada pelo chatbot.
#
# Sempre que novos documentos forem adicionados,
# removidos ou alterados, execute este arquivo novamente.
#
# O processo é:
#
# 1 - Ler todos os documentos
# 2 - Dividir os textos em pedaços
# 3 - Criar os embeddings
# 4 - Salvar a base FAISS
#
# ============================================================


# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import os

from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.vectorstores import FAISS

from leitor_documentos import ler_arquivo

# ============================================================
# CARREGA O ARQUIVO .ENV
# ============================================================

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:

    raise ValueError(

        """
A chave GOOGLE_API_KEY não foi encontrada.

Verifique se existe um arquivo .env
na pasta principal do projeto.

Exemplo:

GOOGLE_API_KEY=sua_chave_aqui
"""

    )

 ============================================================
# CONFIGURAÇÕES
# ============================================================

PASTA_DOCUMENTOS = "documentos"

PASTA_VECTORSTORE = "vectorstore"


# ============================================================
# VERIFICA SE A PASTA DOCUMENTOS EXISTE
# ============================================================

if not os.path.exists(PASTA_DOCUMENTOS):

    raise FileNotFoundError(

        "A pasta 'documentos' não foi encontrada."

    )


# ============================================================
# MODELO DE EMBEDDINGS
# ============================================================

print("Carregando modelo de Embeddings...")

embeddings = GoogleGenerativeAIEmbeddings(

    model="gemini-embedding-001",

    google_api_key=GOOGLE_API_KEY

)


# ============================================================
# DIVISOR DE TEXTO
# ============================================================

"""
Documentos muito grandes são divididos
em pequenos pedaços.

Isso melhora bastante a busca.
"""

quebrador = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=150

)


# ============================================================
# LISTAS E CONTADORES
# ============================================================

todos_os_pedacos = []

total_arquivos = 0

arquivos_processados = 0

arquivos_com_erro = 0


print()

print("=" * 60)

print("INICIANDO LEITURA DOS DOCUMENTOS")

print("=" * 60)


 ============================================================
# PERCORRE TODAS AS PASTAS
# ============================================================

for raiz, pastas, arquivos in os.walk(PASTA_DOCUMENTOS):

    for nome_arquivo in arquivos:

        # Ignora arquivos temporários

        if nome_arquivo.startswith("~$"):

            continue

        if nome_arquivo in ["Thumbs.db", ".DS_Store"]:

            continue

        total_arquivos += 1

        caminho = os.path.join(raiz, nome_arquivo)

        categoria = os.path.basename(raiz)

        print()

        print("-" * 60)

        print(f"Arquivo: {nome_arquivo}")

        print(f"Categoria: {categoria}")

        try:

            documentos = ler_arquivo(caminho)

            if len(documentos) == 0:

                print("Formato não suportado.")

                continue

            # Adiciona a categoria ao documento

            for documento in documentos:

                documento.metadata["categoria"] = categoria

            # Divide o documento em pedaços

            pedacos = quebrador.split_documents(documentos)

            todos_os_pedacos.extend(pedacos)

            arquivos_processados += 1

            print(f"Documentos encontrados : {len(documentos)}")

            print(f"Pedaços criados       : {len(pedacos)}")

        except Exception as erro:

            arquivos_com_erro += 1

            print("Erro ao processar o arquivo.")

            print(erro)


# ============================================================
# VERIFICA SE EXISTEM DOCUMENTOS
# ============================================================

if len(todos_os_pedacos) == 0:

    raise ValueError(

        """
Nenhum documento foi processado.

Verifique:

• Se existem arquivos na pasta documentos.

• Se os formatos são suportados.

• Se houve erro durante a leitura.
"""

    )
